In [75]:
from pathlib import Path
import re

# Data manipulation and cleaning helpers
import pandas as pd
from bs4 import BeautifulSoup
from html import unescape
from unidecode import unidecode

# Tokenization, stop words, and lemmatization
import nltk
from nltk.corpus import stopwords

import spacy
from spacy.cli import download as spacy_download

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
# Vectorization utilities for building the preprocessing pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer


In [76]:
#  Locate CSV files that contain the training texts and labels
DATA_DIR = Path('../../Dataset')
X_train_path = DATA_DIR / 'X_train.csv'
Y_train_path = DATA_DIR / 'Y_train.csv'

# Read both datasets while preserving the shared index for later merging
X_train = pd.read_csv(X_train_path, index_col=0)
Y_train = pd.read_csv(Y_train_path, index_col=0)

In [77]:
# fusion et préparation des colonnes textuelles — consolide les champs 
# utiles pour que chaque produit soit représenté par un texte exploitable

# Combine features and labels, then assemble a single raw text column
train_df = X_train.join(Y_train, how='left')
text_columns = ['designation', 'description']

# Replace missing descriptions/titles with blanks to avoid NaN issues downstream
train_df[text_columns] = train_df[text_columns].fillna('')

# Concatenate the textual fields into one string per product
train_df['text_test'] = train_df[text_columns].agg(' '.join, axis=1)

train_df.head()

,designation,description,productid,imageid,prdtypecode,text_test
0,Olivia: Personalisiertes Notizbuch / 150 Seite...,,3804725264,1263597046,10,Olivia: Personalisiertes Notizbuch / 150 Seite...
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,,436067568,1008141237,2280,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,50,Grand Stylet Ergonomique Bleu Gamepad Nintendo...
3,Peluche Donald - Europe - Disneyland 2000 (Mar...,,50418756,457047496,1280,Peluche Donald - Europe - Disneyland 2000 (Mar...
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,2705,La Guerre Des Tuques Luc a des id&eacute;es de...


In [78]:
# fonctions de nettoyage — centralise les routines qui réduisent le bruit lexical avant la vectorisation

# Helper functions to strip HTML, normalize whitespace, and standardize text
def strip_html(text: str) -> str:
    #Remove HTML tags while keeping readable spacing.
    if not text:
        return ''
    soup = BeautifulSoup(text, 'html.parser')
    return soup.get_text(separator=' ')


def normalize_whitespace(text: str) -> str:
    #Collapse multiple spaces/newlines into a single space.
    return re.sub(r'\s+', ' ', text).strip()


def clean_text(text: str) -> str:
    #Full cleaning pipeline applied before vectorization.
    if text is None:
        text = ''
    text = unescape(text)  # Decode HTML entities like &eacute;
    text = strip_html(text)
    text = text.lower()
    text = unidecode(text)  # Remove accents to harmonize tokens
    text = normalize_whitespace(text)
    return text

In [79]:
# Diagnostic summary — inspect how cleaning affects text length and token counts

def summarize_cleaning_effects(dataframe: pd.DataFrame, sample_size: int = 2000) -> pd.DataFrame:
    #Return descriptive stats comparing raw vs. cleaned text lengths.
    subset = dataframe[['text_test']].copy()
    if sample_size and len(subset) > sample_size:
        subset = subset.sample(sample_size, random_state=42)
    cleaned_series = subset['text_test'].map(clean_text)
    summary_df = pd.DataFrame({
        'original_chars': subset['text_test'].str.len(),
        'clean_chars': cleaned_series.str.len(),
        'original_tokens': subset['text_test'].str.split().map(len),
        'clean_tokens': cleaned_series.str.split().map(len),
    })
    summary_df['delta_chars'] = summary_df['original_chars'] - summary_df['clean_chars']
    summary_df['delta_tokens'] = summary_df['original_tokens'] - summary_df['clean_tokens']
    return summary_df.describe().round(2)

cleaning_summary = summarize_cleaning_effects(train_df)
print('Cleaning impact on character and token counts (sampled rows):')
print(cleaning_summary)


Cleaning impact on character and token counts (sampled rows):
       original_chars  clean_chars  original_tokens  clean_tokens  \
count         2000.00      2000.00          2000.00       2000.00   
mean           626.99       588.99            96.30         95.58   
std            805.92       755.07           124.02        123.21   
min             14.00        13.00             4.00          4.00   
25%             69.00        68.00            12.00         12.00   
50%            357.00       340.00            56.00         56.00   
75%            929.25       887.25           141.00        141.00   
max           9171.00      9061.00          1465.00       1465.00   

       delta_chars  delta_tokens  
count       2000.0       2000.00  
mean          38.0          0.72  
std           91.5          6.11  
min          -10.0        -30.00  
25%            1.0          0.00  
50%            4.0          0.00  
75%           28.0          0.00  
max          837.0        102.00  


In [80]:
# tokenisation et lemmatisation — prépare le découpage et la normalisation
# qui alimentent les vecteurs de caractéristiques

# Download stop-word lists on first run and prepare spaCy lemmatizer
nltk.download('stopwords', quiet=True)

def normalize_stopword_list(words: Iterable[str]) -> Set[str]:
    #Return a normalized set of stop words without accents for filtering.
    return {unidecode(word).lower() for word in words}

french_stopwords = normalize_stopword_list(stopwords.words('french'))
english_stopwords = normalize_stopword_list(stopwords.words('english'))
stopword_set = french_stopwords | english_stopwords

def ensure_spacy_model(model_name: str) -> spacy.language.Language:
    #Load a spaCy model, downloading it when it is not yet available.
    try:
        return spacy.load(model_name, disable=['ner'])
    except OSError:
        spacy_download(model_name)
        return spacy.load(model_name, disable=['ner'])

spacy_model = ensure_spacy_model('fr_core_news_sm')

def tokenize_and_lemmatize(text: str) -> Iterable[str]:
    #Tokenize cleaned text, filter noise, and reduce words to their lemmas.
    if not text:
        return []
    doc = spacy_model(text)
    lemmas = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and len(token) > 2
    ]
    filtered = [lemma for lemma in lemmas if lemma not in stopword_set]
    return filtered



In [81]:
# Diagnostic preview — show original vs. cleaned text and tokens for a representative row

def select_representative_index(dataframe: pd.DataFrame) -> int:
    html_rows = dataframe[dataframe['description'].str.contains('<', na=False, regex=False)]
    if not html_rows.empty:
        return html_rows.index[0]
    return dataframe.index[0]

def preview_preprocessing(idx: int) -> None:
    row = train_df.loc[idx]
    original_text = row['text_test']
    cleaned_text = clean_text(original_text)
    tokens = tokenize_and_lemmatize(cleaned_text)

    print(f'Row index: {idx}')
    print('Designation:')
    print(row['designation'])
    print('\nDescription:')
    print(row['description'])
    print('\nCombined text (first 500 chars):')
    print(original_text[:500])
    print('\nCleaned text (first 500 chars):')
    print(cleaned_text[:500])
    print('\nTokens after lemmatization (first 40):')
    print(tokens[:40])
    print(f'\nOriginal char length: {len(original_text)} | Clean char length: {len(cleaned_text)}')
    print(f'Token count: {len(tokens)}')

sample_index = select_representative_index(train_df)
preview_preprocessing(sample_index)


Row index: 2
Designation:
Grand Stylet Ergonomique Bleu Gamepad Nintendo Wii U - Speedlink Pilot Style

Description:
PILOT STYLE Touch Pen de marque Speedlink est 1 stylet ergonomique pour GamePad Nintendo Wii U.<br> Pour un confort optimal et une précision maximale sur le GamePad de la Wii U: ce grand stylet hautement ergonomique est non seulement parfaitement adapté à votre main mais aussi très élégant.<br> Il est livré avec un support qui se fixe sans adhésif à l'arrière du GamePad<br> <br> Caractéristiques:<br> Modèle: Speedlink PILOT STYLE Touch Pen<br> Couleur: Bleu<br> Ref. Fabricant: SL-3468-BE<br> Compatibilité: GamePad Nintendo Wii U<br> Forme particulièrement ergonomique excellente tenue en main<br> Pointe à revêtement longue durée conçue pour ne pas abîmer l'écran tactile<br> En bonus : Support inclu pour GamePad<br> <span class="vga_style2"><b></b><br>

Combined text (first 500 chars):
Grand Stylet Ergonomique Bleu Gamepad Nintendo Wii U - Speedlink Pilot Style PILOT STYLE

In [82]:
# Frequency-filter configuration for token vocabularies
APPLY_DF_FILTER = True
WORD_MIN_DF = 5
WORD_MAX_DF = 0.8
CHAR_MIN_DF = 5
CHAR_MAX_DF = 0.9


In [83]:
# configuration des vectoriseurs — construit les transformateurs TF-IDF qui convertiront 
# le texte nettoyé en caractéristiques numériques exploitables

# Word-level TF-IDF with lemmatization-aware tokenizer captures vocabulary and bigrams
word_vectorizer = TfidfVectorizer(
    preprocessor=clean_text,
    tokenizer=tokenize_and_lemmatize,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=WORD_MIN_DF if APPLY_DF_FILTER else 1,
    max_df=WORD_MAX_DF if APPLY_DF_FILTER else 1.0,
    sublinear_tf=True,
)

# Character n-gram TF-IDF to capture subword patterns and handle misspellings
char_vectorizer = TfidfVectorizer(
    preprocessor=clean_text,
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=CHAR_MIN_DF if APPLY_DF_FILTER else 1,
    max_df=CHAR_MAX_DF if APPLY_DF_FILTER else 1.0,
    sublinear_tf=True,
)

# Combine word and character features into a single sparse representation
text_vectorizer = FeatureUnion([
    ('word', word_vectorizer),
    ('char', char_vectorizer),
])

# Wrap the feature union in a pipeline that selects the raw text column
text_pipeline = Pipeline([
    ('select_text', FunctionTransformer(lambda df: df['text_test'], validate=False)),
    ('vectorize', text_vectorizer),
])


In [84]:
# éparation et vectorisation des jeux d'entraînement/validation — produit 
# les matrices finales tout en préservant une validation représentative pour l'évaluation

# Separate features/target and create a stratified train/validation split
X_features = train_df[['text_test']].copy()
y_target = train_df['prdtypecode'].copy()

X_train_split, X_valid_split, y_train_split, y_valid_split = train_test_split(
    X_features,
    y_target,
    test_size=0.2,
    random_state=42,
    stratify=y_target,
)

# Fit the text pipeline on the training data and transform both splits
vectorizer_model = text_pipeline.fit(X_train_split, y_train_split)

X_train_vectors = vectorizer_model.transform(X_train_split)
X_valid_vectors = vectorizer_model.transform(X_valid_split)

# Display the resulting sparse matrix shapes for sanity checking
print(f'Train matrix shape: {X_train_vectors.shape}')
print(f'Validation matrix shape: {X_valid_vectors.shape}')

Train matrix shape: (67932, 317321)
Validation matrix shape: (16984, 317321)


In [85]:
# Modèle de classification — entraîne un classifieur linéaire sur les vecteurs TF-IDF

classifier = SGDClassifier(
    loss='log_loss',
    alpha=1e-4,
    penalty='l2',
    max_iter=1000,
    random_state=42,
    n_iter_no_change=5,
    tol=1e-4,
)
classifier.fit(X_train_vectors, y_train_split)

y_train_pred = classifier.predict(X_train_vectors)
y_valid_pred = classifier.predict(X_valid_vectors)

train_accuracy = accuracy_score(y_train_split, y_train_pred)
valid_accuracy = accuracy_score(y_valid_split, y_valid_pred)
macro_f1 = f1_score(y_valid_split, y_valid_pred, average='macro')
weighted_f1 = f1_score(y_valid_split, y_valid_pred, average='weighted')
report = classification_report(y_valid_split, y_valid_pred)

print(f'Accuracy: {valid_accuracy:.3f}')
print(f'F1 Macro: {macro_f1:.3f}')
print(f'F1 Weighted: {weighted_f1:.3f}')
print('Classification Report:')
print(report)


Accuracy: 0.787
F1 Macro: 0.759
F1 Weighted: 0.785
Classification Report:
              precision    recall  f1-score   support

          10       0.39      0.65      0.48       623
          40       0.79      0.50      0.61       502
          50       0.76      0.76      0.76       336
          60       0.99      0.70      0.82       166
        1140       0.77      0.78      0.78       534
        1160       0.86      0.94      0.90       791
        1180       0.94      0.33      0.49       153
        1280       0.66      0.61      0.63       974
        1281       0.74      0.41      0.53       414
        1300       0.81      0.94      0.87      1009
        1301       0.98      0.85      0.91       161
        1302       0.83      0.65      0.73       498
        1320       0.84      0.71      0.77       648
        1560       0.75      0.82      0.78      1015
        1920       0.89      0.91      0.90       861
        1940       0.95      0.63      0.76       161
       